# 01 - 快速上手与核心流程

> **一行代码，万行数据。零配置智能生成，AI 驱动精准调优。**

## 场景导航

| 我的需求 | 推荐 Notebook | 核心 API |
|----------|---------------|----------|
| 快速生成测试数据 | **01 - 快速上手** ← 你在这里 | `fill()` |
| 自定义每列的数据类型 | 02 - 列映射策略 | `columns={}` |
| 选择数据生成引擎 | 03 - 生成器与 Provider | `provider=` |
| 多表关联、外键完整性 | 04 - 数据库与多表关联 | `connect()` + `fill_from_config()` |
| 列之间的派生关系 | 05 - 表达式与约束 | `derive_from` + `expression` |
| YAML 配置驱动、批量生成 | 06 - 配置与 Transform | `fill_from_config()` |
| AI 自动生成配置 | 07 - AI 智能配置 | `sqlseed-ai` 插件 |
| AI 助手操作数据库 | 08 - MCP 服务器 | `mcp-server-sqlseed` |
| 自定义插件扩展 | 09 - 插件与 Hook | `pluggy` |
| 命令行操作 | 10 - CLI 参考 | `sqlseed` CLI |

## 你将学到

- 一行代码填充：`sqlseed.fill()` 的力量
- 零配置智能推断：sqlseed 如何自动选择生成器
- 预览数据：`sqlseed.preview()` 不写入数据库
- 上下文管理器：`sqlseed.connect()` 精细控制
- 核心参数详解：count, provider, seed, batch_size, enrich

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| **→ 01** | **快速上手与核心流程** | **Orchestrator** | **无** |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [ ]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| 核心编排 | `src/sqlseed/core/orchestrator.py` | `DataOrchestrator.fill_table()` |

> 对应架构图: [§2 核心编排流程（fill_table 执行链路）](../docs/architecture.zh-CN.md#2-核心编排流程fill_table-执行链路)

## 1. 一行代码，万行数据 — sqlseed 的力量

sqlseed 的核心理念：**一行代码即可生成海量测试数据**。无需编写生成脚本、无需维护 SQL fixtures — sqlseed 自动推断表结构，智能选择生成策略，流式写入数据库。

```python
result = sqlseed.fill("app.db", table="users", count=100_000)
```

In [19]:
# 先填充父表（organizations），再填充子表（members）
# sqlseed 需要父表数据来解析外键引用
fill(str(db_path), table="organizations", count=5)

# 一行代码，生成 100 条 member 数据
result = fill(str(db_path), table="members", count=100)
print(result)
# → GenerationResult(table=members, count=100, elapsed=..., speed=...)

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/100 [00:00<?, ?it/s]

GenerationResult(table=members, count=100, elapsed=0.05s, speed=1897.96 rows/s)


就这一行 — sqlseed 完成了：

1. **读取 schema** — 自动检测 `members` 表的列、类型、约束
2. **智能映射** — `name` → 真实姓名，`email` → 邮箱地址，`member_no` → 唯一编号
3. **流式写入** — 批量插入 100 行，自动处理 UNIQUE 约束
4. **返回结果** — `GenerationResult` 包含行数、耗时、速度等信息

## 2. 查看数据库结构

在生成数据之前，先了解数据库有哪些表和列。

In [20]:
import sqlite3

conn = sqlite3.connect(str(db_path))
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()]
print(f"Tables ({len(tables)}): {tables}")

for table in tables:
    cols = conn.execute(f"PRAGMA table_info({table})").fetchall()
    col_names = [c[1] for c in cols]
    print(f"  {table}: {col_names}")

conn.close()

Tables (9): ['attachments', 'members', 'organizations', 'projects', 'reviews', 'sqlite_sequence', 'tags', 'task_tags', 'tasks']
  attachments: ['attachment_id', 'task_id', 'file_name', 'file_data', 'file_size', 'uploaded_at']
  members: ['member_id', 'member_no', 'name', 'email', 'phone', 'org_code', 'is_active', 'balance', 'avatar', 'registered_at', 'address']
  organizations: ['org_code', 'name', 'parent_code', 'description', 'is_active', 'member_count', 'created_at']
  projects: ['project_id', 'project_no', 'short_code', 'name', 'org_code', 'budget', 'task_count', 'is_public', 'is_archived', 'created_at', 'description']
  reviews: ['review_id', 'task_id', 'member_id', 'rating', 'content', 'created_at']
  sqlite_sequence: ['name', 'seq']
  tags: ['tag_id', 'name', 'color', 'usage_count']
  task_tags: ['task_id', 'tag_id']
  tasks: ['task_id', 'project_id', 'assignee_id', 'title', 'priority', 'status', 'is_completed', 'comment_count', 'estimated_hours', 'due_at', 'completed_at', 'crea

## 3. 零配置填充详解

上面的 `fill()` 一行代码背后，sqlseed 做了很多工作。让我们详细看看：

In [21]:
result = fill(str(db_path), table="members", count=10)
print(result)

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

GenerationResult(table=members, count=10, elapsed=0.03s, speed=318.32 rows/s)


### GenerationResult 详解

返回的 `GenerationResult` 包含丰富的执行信息：

In [22]:
print(f"表名: {result.table_name}")
print(f"插入行数: {result.count}")
print(f"耗时: {result.elapsed:.3f}s")
print(f"速度: {result.rows_per_second:.2f} rows/s")
print(f"批次数: {result.batch_count}")
print(f"错误: {result.errors}")

表名: members
插入行数: 10
耗时: 0.031s
速度: 318.32 rows/s
批次数: 10
错误: []


### 查看生成的数据

In [23]:
conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT member_id, member_no, name, email, org_code FROM members LIMIT 5").fetchall()

# 表格化输出
print(f"{'ID':>4s}  {'member_no':<20s}  {'name':<20s}  {'email':<30s}  {'org_code':<10s}")
print('-' * 90)
for row in rows:
    print(f"{row[0]:>4d}  {row[1]:<20s}  {row[2]:<20s}  {row[3]:<30s}  {row[4]:<10s}")
conn.close()

  ID  member_no             name                  email                           org_code  
------------------------------------------------------------------------------------------
   1  nPmDMlFXPgF4ppH       Clementina Kemp       likely1848@protonmail.com       4CPvyCA   
   2  IB7jO7eWeiHlu         Lashawna Henson       hugo2033@example.org            UmTrNDj   
   3  BVUh8MSe              Vannesa Lane          cgi2077@yandex.com              sr7EGr7   
   4  btjym                 Walter Alford         yes1828@example.org             UmTrNDj   
   5  JcEVy2ajAR24          Cesar Wong            surfing2038@gmail.com           YdcroT    


## 4. 预览数据（不写入数据库）

`sqlseed.preview()` 生成数据但**不写入数据库**，适合调试和验证映射结果。

In [24]:
# 先填充 projects 表 (需要 organizations 已有数据)
fill(str(db_path), table="projects", count=5)

# 查看已生成的项目信息
conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, name, org_code FROM projects").fetchall()
print(f"{'project_no':<12s}  {'short_code':<8s}  {'name':<25s}  {'org_code':<10s}")
print('-' * 58)
for row in rows:
    print(f"{row[0]:<12s}  {str(row[1] or ''):<8s}  {row[2]:<25s}  {row[3]:<10s}")
conn.close()

# preview 生成新数据(不写入), 可用 columns 覆盖生成策略
print("\npreview 生成新数据 (columns 覆盖):")
rows = preview(str(db_path), table="projects", count=3,
               columns={"project_no": {"type": "pattern", "regex": PRJ_PATTERN},
                        "name": {"type": "company"}})
for row in rows:
    print(f"  {row.get('project_no'):<12s}  {row.get('name')}")

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

project_no    short_code  name                       org_code  
----------------------------------------------------------
iMTTLoG0vOXYf  FO1wmL9F3  Lesley Stuart              YdcroT    
gANUHUxQNmxluGNbAf  QOKb4z    Marcos Stone               4CPvyCA   
cQ7WfHRj2z7   nZRqs2VC  Ron Nielsen                UmTrNDj   
ngeQ          evtGFP    Felipe Burnett             TFCTkQU9LZq7
hukbS rYnIuCFR cGKn  lJFdWFSeP  Philip Sweet               UmTrNDj   

preview 生成新数据 (columns 覆盖):
  PRJ-132130    American Eagle Outfitters
  PRJ-491990    AirTran Holdings
  PRJ-241327    Gamma Gas


## 5. 上下文管理器

`sqlseed.connect()` 返回 `DataOrchestrator` 上下文管理器，适合需要多次填充的场景。它会在退出时自动清理资源。

In [25]:
with connect(str(db_path), provider="mimesis", locale="en") as orch:
    r1 = orch.fill_table("tags", count=10)
    r2 = orch.fill_table("tasks", count=50)
    print(f"Tags: {r1.count} rows in {r1.elapsed:.3f}s")
    print(f"Tasks: {r2.count} rows in {r2.elapsed:.3f}s")

Generating tags:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/50 [00:00<?, ?it/s]

Tags: 10 rows in 0.033s
Tasks: 50 rows in 0.036s


## 6. 核心参数详解

### 6.1 count — 生成行数

控制生成的数据量。对于有 UNIQUE 约束的列，sqlseed 会自动回溯求解确保不重复。

In [26]:
import sqlite3

conn = sqlite3.connect(str(db_path))
existing_codes = [r[0] for r in conn.execute("SELECT org_code FROM organizations").fetchall()]

result = fill(str(db_path), table="organizations", count=3,
              columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                       "parent_code": {"type": "choice", "choices": existing_codes}})
print(f"生成 {result.count} 条组织数据 ({result.elapsed:.3f}s, {result.rows_per_second:.0f} rows/s)")

rows = conn.execute("SELECT org_code, name, parent_code FROM organizations").fetchall()
print(f"\n{'org_code':<12s}  {'name':<25s}  {'parent_code':<12s}")
print('-' * 52)
for row in rows:
    parent = row[2] or '(root)'
    print(f"{row[0]:<12s}  {row[1]:<25s}  {parent:<12s}")
conn.close()

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

生成 3 条组织数据 (0.022s, 134 rows/s)

org_code      name                       parent_code 
----------------------------------------------------
UmTrNDj       Samira Mayer               673557      
4CPvyCA       Lisandra Snider            56685       
YdcroT        Tamiko Joyner              77537       
sr7EGr7       Darryl Townsend            299321      
TFCTkQU9LZq7  Bibi West                  491709      
ORG-7773      Lee Serrano                4CPvyCA     
ORG-1238      Hiedi Lamb                 TFCTkQU9LZq7
ORG-1246      Herma Lynch                TFCTkQU9LZq7


### 6.2 provider — 数据提供者

sqlseed 支持三种 Provider，按丰富度降级：

| Provider | 依赖 | 生成质量 | 速度 |
|----------|------|----------|------|
| `mimesis` | mimesis | 最高（本地化、语义丰富） | 快 |
| `faker` | faker | 高（方法多、社区大） | 中 |
| `base` | 无 | 基础（随机字符串/数字） | 最快 |

In [27]:
for provider_name in ["base", "faker", "mimesis"]:
    rows = preview(str(db_path), table="members", count=2, provider=provider_name)
    sep = "=" * 60
    print()
    print(sep)
    print(f"  Provider: {provider_name}")
    print(sep)
    for row in rows:
        addr = str(row.get("address", "N/A"))[:40]
        print(f"  name={row.get('name', 'N/A'):<20s} email={row.get('email', 'N/A'):<30s}")
        print(f"  phone={row.get('phone', 'N/A'):<20s} address={addr}")


  Provider: base
  name=Joseph Robinson      email=margaret.martinez185@sample.dev
  phone=778-269-1431         address=2059 Washington Ave, Salem, PA
  name=David Rodriguez      email=deborah.ramirez925@mail.net   
  phone=990-912-2228         address=9633 Park Rd, Georgetown, CA

  Provider: faker
  name=Erin Mcclain         email=jonathangarrett@example.com   
  phone=224-573-7761x513     address=9342 Hill Skyway Apt. 282, South Jamesbu
  name=Jenna Heath          email=loricastillo@example.com      
  phone=001-830-771-7590x268 address=22765 Christopher Crossing, Port Ronaldb

  Provider: mimesis
  name=Emely William        email=shades1809@example.com        
  phone=+1-401-424-2727      address=46 Service Avenue
  name=Mendy Henson         email=wage1919@yandex.com           
  phone=+1-405-418-2605      address=1116 Albatross Canyon


### 6.3 seed — 可复现性

设置相同的 seed 可以确保每次生成相同的数据，适合测试和调试。

In [28]:
rows_a = preview(str(db_path), table="members", count=3, seed=42)
rows_b = preview(str(db_path), table="members", count=3, seed=42)
rows_c = preview(str(db_path), table="members", count=3, seed=99)

names_a = [r["name"] for r in rows_a]
names_b = [r["name"] for r in rows_b]
names_c = [r["name"] for r in rows_c]

print(f"seed=42 (run 1): {names_a}")
print(f"seed=42 (run 2): {names_b}")
print(f"seed=99:        {names_c}")
print(f"\nseed=42 可复现: {names_a == names_b}")
print(f"seed 不同:      {names_a != names_c}")

seed=42 (run 1): ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
seed=42 (run 2): ['Anthony Reilly', 'Jaye Hunt', 'Randy Lynch']
seed=99:        ['Myung Cote', 'Lyle Shields', 'Nicky Craig']

seed=42 可复现: True
seed 不同:      True


### 6.4 batch_size — 批量写入大小

控制每次写入数据库的行数。较大的 batch_size 提高写入性能，但占用更多内存。

- 默认值：5000
- 小数据量（<1000 行）：batch_size 影响不大
- 大数据量（>10K 行）：适当增大 batch_size 可提升速度

In [29]:
# batch_size 控制每批写入行数, 默认 5000
# 内部会自动调整以保证进度条平滑显示
r = fill(str(db_path), table="members", count=50)
print(f"50 rows: {r.elapsed:.3f}s, {r.rows_per_second:.0f} rows/s")

r = fill(str(db_path), table="members", count=500)
print(f"500 rows: {r.elapsed:.3f}s, {r.rows_per_second:.0f} rows/s")

Generating members:   0%|          | 0/50 [00:00<?, ?it/s]

50 rows: 0.035s, 1440 rows/s


Generating members:   0%|          | 0/500 [00:00<?, ?it/s]

500 rows: 0.117s, 4272 rows/s


### 6.5 enrich — 智能增强模式

当数据库中已有部分数据时，`enrich=True` 会让 sqlseed 分析现有数据的模式（如枚举值、值域范围），并在生成新数据时保持一致。

详见 architecture.md §2 enrich 流程

In [30]:
import sqlite3

# enrich 演示: 先查看 organizations 表中已有数据

conn = sqlite3.connect(str(db_path))
before = conn.execute(COUNT_ORG_SQL).fetchone()[0]
orgs = conn.execute("SELECT org_code, name FROM organizations").fetchall()
print(f"已有数据: {before} 个组织")
for r in orgs:
    print(f"  {r[0]:<12s} {r[1]}")

# 获取已有的 org_code 作为 parent_code 的候选值
existing_codes = [r[0] for r in orgs]

# enrich=True: 分析现有数据模式, 新增数据保持一致
r2 = fill(str(db_path), table="organizations", count=5, enrich=True,
          columns={"org_code": {"type": "pattern", "regex": ORG_PATTERN},
                   "parent_code": {"type": "choice", "choices": existing_codes}})
after = conn.execute(COUNT_ORG_SQL).fetchone()[0]
new_rows = conn.execute("SELECT org_code, name, parent_code FROM organizations ORDER BY rowid DESC LIMIT 5").fetchall()
print(f"\nenrich 新增 {r2.count} 个 ({before} -> {after}):")
for r in new_rows:
    print(f"  {r[0]:<12s} {r[1]:<22s} parent={r[2] or ''}")
conn.close()


已有数据: 8 个组织
  UmTrNDj      Samira Mayer
  4CPvyCA      Lisandra Snider
  YdcroT       Tamiko Joyner
  sr7EGr7      Darryl Townsend
  TFCTkQU9LZq7 Bibi West
  ORG-7773     Lee Serrano
  ORG-1238     Hiedi Lamb
  ORG-1246     Herma Lynch


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]


enrich 新增 5 个 (8 -> 13):
  ORG-8195     Kelsie Mayer           parent=sr7EGr7
  ORG-4496     Loma Woods             parent=YdcroT
  ORG-4665     Brendon Dodson         parent=UmTrNDj
  ORG-3885     Cassy Stevens          parent=YdcroT
  ORG-2094     Erik Valenzuela        parent=YdcroT


### 6.6 clear_before — 清空后填充

清除表中已有数据后再填充。注意：如果有外键引用，需要先填充被引用表。

In [31]:
result = fill(str(db_path), table="tags", count=8, clear_before=True)
print(f"Cleared and refilled: {result.count} rows")

conn = sqlite3.connect(str(db_path))
count = conn.execute("SELECT COUNT(*) FROM tags").fetchone()[0]
print(f"Current row count: {count}")
conn.close()

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

Cleared and refilled: 8 rows
Current row count: 8


## 7. 自定义列映射

通过 `columns` 参数可以覆盖自动推断的列映射：

In [32]:
result = fill(
    str(db_path),
    table="members",
    count=5,
    columns={
        "name": {"type": "name"},               # 真实姓名
        "email": {"type": "email"},             # 邮箱地址
        "phone": {"type": "phone"},             # 电话号码
        "balance": {"type": "float", "min_value": 100.0, "max_value": 500.0},  # 指定范围
        "is_active": {"type": "boolean"},       # 布尔值
    },
)
print(f"自定义映射: {result.count} rows")

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT name, email, phone, balance, is_active FROM members ORDER BY member_id DESC LIMIT 5").fetchall()  # noqa: E501
print(f"\n{'name':<18s}  {'email':<28s}  {'phone':<18s}  {'balance':>8s}  {'active':>6s}")
print('-' * 85)
for row in rows:
    print(f"{row[0]:<18s}  {row[1]:<28s}  {row[2]:<18s}  {row[3]:>8.2f}  {row[4]!s:>6s}")
conn.close()

Generating members:   0%|          | 0/5 [00:00<?, ?it/s]

自定义映射: 5 rows

name                email                         phone                balance  active
-------------------------------------------------------------------------------------
Raymonde Small      benjamin2050@yandex.com       +13097970592          317.87       0
Ashlea Calhoun      purpose1916@yandex.com        +1-480-695-0643       188.75       1
Nakisha Hansen      ceramic1929@yandex.com        +15151737329          184.20       1
Doreatha Diaz       treat1884@yandex.com          +1-508-366-9905       437.64       0
Bernardina Hernandez  true2061@example.org          +1-727-572-3530       304.01       0


## 🎯 enrich 模式详解

`enrich=True` 时，sqlseed 会自动检测**枚举列**（如 `status`、`*_type`、`is_*` 等），即使这些列有 DEFAULT 值或可 NULL，也会生成有意义的枚举值而非跳过。

EnrichmentEngine 使用 19 种枚举列名模式和基数比计算来识别枚举列。详见 [02-column-mapping](02-column-mapping.ipynb)。

In [33]:
import sqlite3

# enrich 模式详解: 在 organizations 表上演示
# 先填充基准数据
fill(str(db_path), table="organizations", count=3, seed=42,
     columns={"org_code": {"type": "pattern", "regex": r"ENR-\d{4}"}})

conn = sqlite3.connect(str(db_path))
before = conn.execute(COUNT_ORG_SQL).fetchone()[0]
existing_codes = [r[0] for r in conn.execute("SELECT org_code FROM organizations").fetchall()]
print(f"enrich 前: {before} 个组织")

# enrich 新增组织, 自动保持已有数据模式
r2 = fill(str(db_path), table="organizations", count=3, enrich=True,
          columns={"org_code": {"type": "pattern", "regex": r"ENR-\d{4}"},
                   "parent_code": {"type": "choice", "choices": existing_codes}})
after = conn.execute(COUNT_ORG_SQL).fetchone()[0]
new_rows = conn.execute("SELECT org_code, name, parent_code FROM organizations ORDER BY rowid DESC LIMIT 3").fetchall()
print(f"enrich 后: {after} 个组织 (新增 {r2.count})")
print("\nenrich 新增:")
for r in new_rows:
    print(f"  {r[0]:<12s} {r[1]:<22s} parent={r[2] or ''}")
conn.close()

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

enrich 前: 16 个组织


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

enrich 后: 19 个组织 (新增 3)

enrich 新增:
  ENR-7457     Petronila Workman      parent=YdcroT
  ENR-3083     Denyse Hampton         parent=ORG-2094
  ENR-2881     Colby Valenzuela       parent=TFCTkQU9LZq7


## 📋 fill_from_config 入门

最简 YAML 配置 + `fill_from_config()` 一行调用即可批量填充多表。详见 [06-config-deep-dive](06-config-deep-dive.ipynb)。

In [34]:
from pathlib import Path
from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

simple_config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name="organizations", count=3),
    ]
)
config_path = Path("_quickstart_config.yaml")
save_config(simple_config, str(config_path))

results = fill_from_config(str(config_path))
for r in results:
    print(f"  {r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

  organizations: 3 rows in 0.022s


## ⚠️ 错误处理

常见错误及处理方式：

In [35]:
r1 = fill(str(db_path), table="nonexistent_table", count=1)
if r1.errors:
    print(f"表不存在 (内部捕获): {r1.errors[0]}")

try:
    fill(str(db_path), table="organizations", count=-1)
except ValueError as e:
    print(f"参数错误 (抛出异常): {e}")

try:
    fill(str(db_path), table="; DROP TABLE organizations; --", count=1)
except ValueError as e:
    print(f"SQL 注入防护: {e}")

2026-05-05T23:06:42.363028Z [error    ] Failed to fill table           error=OperationalError('no such table: nonexistent_table') table_name=nonexistent_table


表不存在 (内部捕获): no such table: nonexistent_table
参数错误 (抛出异常): count must be greater than 0, got -1


2026-05-05T23:06:42.451253Z [warning  ] Table name '; DROP TABLE organizations; --' contains special characters and will be quoted


SQL 注入防护: SQL identifier '; DROP TABLE organizations; --' contains dangerous characters and is rejected


## 8. 总结

| API | 用途 | 写入数据库 |
|-----|------|:----------:|
| `fill()` | 一行代码填充 | ✅ |
| `preview()` | 预览不写入 | ❌ |
| `connect()` | 上下文管理器 | ✅ |
| `fill_from_config()` | YAML/JSON 批量填充 | ✅ |

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `count` | 1000 | 生成行数 |
| `provider` | mimesis | 数据提供者 (mimesis/faker/base) |
| `seed` | None | 随机种子，设置后可复现 |
| `batch_size` | 5000 | 批量写入大小 |
| `enrich` | False | 智能增强模式 |
| `clear_before` | False | 清空后填充 |
| `locale` | en_US | 本地化设置 |

**下一步**: [02-column-mapping.ipynb](02-column-mapping.ipynb) — 深入了解 9 级策略链

In [36]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
